In [88]:
from lakehouse import LakeConfig, connect

lake = LakeConfig.from_env()
con = connect()   # DuckDB already authenticated on MinIO (minio:9000)

order_cab = lake.uri("bronze", "mysql", "order_cab") + "/*/*.parquet"
order_det = lake.uri("bronze", "mysql", "order_det") + "/*/*.parquet"
product = lake.uri("bronze", "mysql", "product") + "/*/*.parquet"
events = lake.uri("bronze", "mysql", "events") + "/*/*.parquet"
sessions = lake.uri("bronze", "mysql", "sessions") + "/*/*.parquet"
emails = lake.uri("bronze", "mysql", "emails") + "/*/*.parquet"
customers = lake.uri("bronze", "mysql", "customers") + "/*/*.parquet"
paid_media = lake.uri("bronze", "mysql", "paid_media") + "/*/*.parquet"


# Tipagem de dados

In [4]:
from lakehouse import LakeConfig, connect

lake = LakeConfig.from_env()
con = connect()   # DuckDB already authenticated on MinIO (minio:9000)

# Lista de tabelas presentes no diretório bronze/mysql
tabelas = [
    "customers",
    "emails",
    "events",
    "order_cab",
    "order_det",
    "paid_media",
    "product",
    "sessions"
]

for tabela in tabelas:
    print(f"\n{'='*60}")
    print(f"TABELA: {tabela.upper()}")
    print(f"{'='*60}")
    
    # Gera a URI para ler todos os arquivos parquet da tabela atual
    tabela_uri = lake.uri("bronze", "mysql", tabela) + "/*/*.parquet"
    
    # 1. Exibe as colunas e os tipos de dados do dataset
    print(f"--- Estrutura (Tipagem de Dados) ---")
    con.sql(f"DESCRIBE SELECT * FROM read_parquet('{tabela_uri}')").show()
    
    # 2. Exibe 5 valores (linhas) aleatórios da tabela
    # O DuckDB permite usar ORDER BY random() para buscar amostras
    print(f"--- 5 Linhas Aleatórias ---")
    df = con.sql(f"SELECT * FROM read_parquet('{tabela_uri}') ORDER BY random() LIMIT 5").df()
    print(df)


TABELA: CUSTOMERS
--- Estrutura (Tipagem de Dados) ---
┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ CUSTOMER_ID         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ CREATION_DATE       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ SUBSCRIPTION_DATE   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ UNSUBSCRIPTION_DATE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ COUNTRY             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ BIRTHDAY            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ SUBSCRIPTION_SOURCE │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ EMAIL_DOMAIN        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ _inges

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   CUSTOMER_ID_FK  SEND_DATE_SFK  SENT  OPENS  CLICKS  UNSUBS  \
0         1650387       20240922     1    NaN     NaN     NaN   
1       159652511       20240728     1    NaN     NaN     NaN   
2       115029902       20240208     1    NaN     NaN     NaN   
3       160315100       20250220     1    1.0     NaN     NaN   
4       160209796       20241007     1    1.0     NaN     NaN   

         _ingested_at _source_system    _source_entity  \
0 2026-10-17 20:29:27          mysql  parfois.emails_3   
1 2026-10-17 20:29:27          mysql  parfois.emails_3   
2 2026-10-17 20:24:01          mysql  parfois.emails_1   
3 2026-10-17 20:34:35          mysql  parfois.emails_5   
4 2026-10-17 20:29:46          mysql  parfois.emails_4   

                              _batch_id         _load_id ingestion_date  
0  scheduled__2026-10-13T05_00_00_00_00  20261017T202927     2026-10-13  
1  scheduled__2026-10-13T05_00_00_00_00  20261017T202927     2026-10-13  
2  scheduled__2026-10-13T05_00_00_00_0

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   DATE_SFK TIMESTAMP    CHANNEL_DESC COUNTRY_COD  CLIENT_SFK  CUSTOMER_ID_FK  \
0  20251031  13:01:41  Organic Search          ES    82298716       160790060   
1  20251127  07:17:39      Social Ads          ES    86782838              -1   
2  20251227  18:04:25     Paid Search          ES    89715230         2584964   
3  20251122  21:22:53     Paid Search          FR    83984331              -1   
4  20250120  14:44:03     Paid Search          ES    40995835              -1   

   SESSION_SFK PLATFORM_DESC  ORDER_ID  USER_TYPE_SFK        _ingested_at  \
0    226164816           Web       NaN              1 2026-10-17 20:45:56   
1    236134870           Web       NaN              2 2026-10-17 20:45:56   
2    248391662           Web       NaN              1 2026-10-17 20:45:56   
3    232776063           Web       NaN              2 2026-10-17 20:45:56   
4     56181391           Web       NaN              2 2026-10-17 20:43:13   

  _source_system      _source_entity              

# Cleaning

## SPK e SFK

In [21]:
from lakehouse import LakeConfig, connect

lake = LakeConfig.from_env()
con = connect()   # DuckDB already authenticated on MinIO (minio:9000)

order_cab = lake.uri("bronze", "mysql", "order_cab") + "/*/*.parquet"
order_det = lake.uri("bronze", "mysql", "order_det") + "/*/*.parquet"
product = lake.uri("bronze", "mysql", "product") + "/*/*.parquet"


# Columns and types of a dataset
df_T = con.sql(f"""

-- Busca o que tem na A e não tem na B
SELECT PROD_SPK AS produto_divergente, 'Somente na Tabela A' AS origem
FROM read_parquet('{product}') a
WHERE NOT EXISTS (
    SELECT 1 
    FROM read_parquet('{order_det}') b 
    WHERE b.PROD_SFK = a.PROD_SPK
)

UNION ALL

-- Busca o que tem na B e não tem na A
SELECT PROD_SFK AS produto_divergente, 'Somente na Tabela B' AS origem
FROM read_parquet('{order_det}')  b
WHERE NOT EXISTS (
    SELECT 1 
    FROM read_parquet('{product}')  a 
    WHERE a.PROD_SPK = b.PROD_SFK
);
""").df()

df_T

,produto_divergente,origem
0,6181247,Somente na Tabela A
1,6181314,Somente na Tabela A
2,6181321,Somente na Tabela A
3,6181351,Somente na Tabela A
4,6181352,Somente na Tabela A
...,...,...
188997,6059455,Somente na Tabela B
188998,5719639,Somente na Tabela B
188999,6059455,Somente na Tabela B
189000,6059455,Somente na Tabela B


In [22]:
# Busca os produtos que estão presentes tanto na Tabela A quanto na Tabela B
df_T = con.sql(f"""
SELECT a.PROD_SPK AS produto_correspondente, 'Em ambas as tabelas' AS origem
FROM read_parquet('{product}') a
WHERE EXISTS (
    SELECT 1 
    FROM read_parquet('{order_det}') b 
    WHERE b.PROD_SFK = a.PROD_SPK
);
""").df()

df_T

,produto_correspondente,origem
0,6181340,Em ambas as tabelas
1,6181469,Em ambas as tabelas
2,6181518,Em ambas as tabelas
3,6181519,Em ambas as tabelas
4,6182438,Em ambas as tabelas
...,...,...
69066,5798093,Em ambas as tabelas
69067,5798177,Em ambas as tabelas
69068,5798325,Em ambas as tabelas
69069,5798502,Em ambas as tabelas


In [25]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select * 
FROM read_parquet('{product}')
WHERE PROD_SPK = 6181321

""").df()

df_T

,PROD_SPK,PROD_COD,CAT_DES_EN,GFA_DES_EN,GFS_DES_EN,COLLECTION_TYPE,SEA_COD,ACTIVE_RECORD,FASHIONTYPE,CAT_ACTIVE,TAR_DES,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,6181321,246194_MMU,Hair Accessories,Headband,Others,Exclusive Online,261,1,Capsule,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13


## Duplicatas Order_Det

In [11]:
# Retorna a quantidade total de linhas que são duplicadas usando o DuckDB
qtd_duplicadas_query = f"""
    SELECT 
        (SELECT count(*) FROM read_parquet('{order_det}')) 
        - 
        (SELECT count(*) FROM (SELECT DISTINCT * FROM read_parquet('{order_det}'))) AS qtd_duplicadas
"""

# Executa a query e pega apenas o valor final
resultado = con.sql(qtd_duplicadas_query).df()
qtd_duplicadas = resultado['qtd_duplicadas'].iloc[0]
print(qtd_duplicadas)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

14810


In [10]:
contagem = f"""
    SELECT count(*)
    FROM read_parquet('{order_det}')
    
"""

# Executa a query e pega apenas o valor final
resultado = con.sql(contagem).df()
#qtd_duplicadas = resultado['qtd_duplicadas'].iloc[0]
print(resultado)

   count_star()
0       9049802


In [12]:
query_duplicatas_negocio = f"""
SELECT 
    ORDER_ID,
    TYPE,
    PROD_SFK,
    MOV_DATE_SFK,
    COUNT(*) as qtd_repeticoes
FROM read_parquet('{order_det}')
GROUP BY 
    ORDER_ID, 
    TYPE, 
    PROD_SFK, 
    MOV_DATE_SFK
HAVING COUNT(*) > 1
ORDER BY ORDER_ID;
"""
df_duplicatas_negocio = con.execute(query_duplicatas_negocio).df()
display(df_duplicatas_negocio)

,ORDER_ID,TYPE,PROD_SFK,MOV_DATE_SFK,qtd_repeticoes
0,900003752829,RETURN,5778521,20240219,2
1,900003881840,RETURN,5853273,20240201,4
2,900003961238,RETURN,5780076,20240201,2
3,900004014184,RETURN,5797058,20240117,2
4,900004018718,RETURN,5795465,20240125,2
...,...,...,...,...,...
11936,900008494991,SALE,6166091,20251230,2
11937,900008495711,SALE,6141405,20251230,2
11938,900008496846,SALE,6182424,20251231,3
11939,900008497355,SALE,6182420,20251231,2


In [23]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select * 
FROM read_parquet('{order_det}')
WHERE ORDER_ID = '900003881840'

""").df()

df_T

,TYPE,ORDER_ID,PROD_SFK,PROD_QTY,MOV_DATE_SFK,AMOUNT_FIXED_RATE_W_DISC_WO_VAT,AMOUNT_FIXED_RATE_WO_DISC_WO_VAT,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,RETURN,900003881840,5853273,-1.0,20240201,-11.21466,-11.21466,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
1,RETURN,900003881840,5853273,-1.0,20240201,-11.21466,-11.21466,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
2,RETURN,900003881840,5853273,-1.0,20240201,-11.21466,-11.21466,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
3,RETURN,900003881840,5853273,-1.0,20240201,-11.21466,-11.21466,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13


In [22]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select * 
FROM read_parquet('{order_cab}')
WHERE ORDER_ID = '900003881840'

""").df()

df_T

,ORDER_ID,ORDER_DATE,ORDER_ORIGIN,ORDER_CHANEL_TYPE,CUSTOMER_TYPE,CUSTOMER_ID_FK,MARK_COD,FULFILLMENT_STATUS_ID,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,900003881840,2023-09-09 17:45:13,PARFOIS,APP,Registered,159169317,Online IT,None,2026-10-17 20:49:46,mysql,parfois.order_cab,scheduled__2026-10-13T05_00_00_00_00,20261017T204946,2026-10-13
1,900003881840,2023-09-09 17:45:13,PARFOIS,APP,Registered,159169317,Online IT,None,2026-10-17 20:49:46,mysql,parfois.order_cab,scheduled__2026-10-13T05_00_00_00_00,20261017T204946,2026-10-13
2,900003881840,2023-09-09 17:45:13,PARFOIS,APP,Registered,159169317,Online IT,None,2026-10-17 20:49:46,mysql,parfois.order_cab,scheduled__2026-10-13T05_00_00_00_00,20261017T204946,2026-10-13
3,900003881840,2023-09-09 17:45:13,PARFOIS,APP,Registered,159169317,Online IT,None,2026-10-17 20:49:46,mysql,parfois.order_cab,scheduled__2026-10-13T05_00_00_00_00,20261017T204946,2026-10-13


## Events 

In [25]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select * 
FROM read_parquet('{events}')

""").df()

df_T

,MARK_COD,MARK_DES,EVENT,START_DATE,END_DATE,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,Online PT,parfois.com,Clearance,2024-01-01,2024-02-28,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
1,Online BE,parfois.com,Clearance,2024-01-02,2024-02-28,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
2,Online IT,parfois.com,Clearance,2024-01-04,2024-02-28,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
3,Online ES,parfois.com,Get your basket ready,2024-01-05,2024-01-05,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
4,Online ES,parfois.com,Clearance,2024-01-06,2024-02-28,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
...,...,...,...,...,...,...,...,...,...,...,...
805,Online PT,parfois.com,Free Shipping,2025-12-27,2025-12-28,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
806,Online IT,parfois.com,Free Shipping,2025-12-27,2025-12-28,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
807,Online ES,parfois.com,Last Days Reyes,2025-12-27,2025-12-30,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
808,Online ES,parfois.com,Last Hours Reyes,2025-12-29,2025-12-30,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13


In [39]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select DISTINCT EVENT
FROM read_parquet('{events}')
WHERE EVENT LIKE '%Christmas'

""").df()

df_T

,EVENT
0,Last Days Christmas
1,Last Hours Christmas
2,Last Days Christmas
3,Last Hours Christmas


## Sessions sem Order_ID encontrado

In [42]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{sessions}')
limit (100)

""").df()

df_T

,DATE_SFK,TIMESTAMP,CHANNEL_DESC,COUNTRY_COD,CLIENT_SFK,CUSTOMER_ID_FK,SESSION_SFK,PLATFORM_DESC,ORDER_ID,USER_TYPE_SFK,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,20240615,16:12:54,Paid Search,ES,5183394,-1,520993,Web,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
1,20240607,20:57:11,Paid Search,PT,21402013,2112290,521002,Web,9.000049e+11,1,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
2,20240627,12:02:12,Social Ads,PT,59735674,187114,521027,APP,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
3,20240611,12:53:52,Email,ES,51591873,-1,521030,Web,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
4,20240613,07:53:56,Direct,PT,33892499,-1,521086,APP,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,20240410,10:48:36,Paid Search,ES,52033618,-1,237236,Web,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
96,20240406,17:33:44,Social,ES,20245187,-1,237243,Web,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
97,20240415,15:55:47,Paid Search,FR,52418124,-1,237248,Web,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
98,20240424,04:44:17,Direct,ES,54816411,169097,237314,APP,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13


In [43]:


query_sessoes_sem_pedidos = f"""
SELECT 
    * REPLACE (CAST(CAST(ORDER_ID AS BIGINT) AS VARCHAR) AS ORDER_ID)
FROM read_parquet('{sessions}')
WHERE ORDER_ID IS NOT NULL
  AND CAST(CAST(ORDER_ID AS BIGINT) AS VARCHAR) NOT IN (
      SELECT ORDER_ID 
      FROM read_parquet('{order_cab}')
      WHERE ORDER_ID IS NOT NULL
  )
  AND CAST(CAST(ORDER_ID AS BIGINT) AS VARCHAR) NOT IN (
      SELECT ORDER_ID 
      FROM read_parquet('{order_det}')
      WHERE ORDER_ID IS NOT NULL
  );
"""

# Executa e converte para DataFrame no Jupyter
df_sessoes_sem_pedidos = con.execute(query_sessoes_sem_pedidos).df()
display(df_sessoes_sem_pedidos)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,DATE_SFK,TIMESTAMP,CHANNEL_DESC,COUNTRY_COD,CLIENT_SFK,CUSTOMER_ID_FK,SESSION_SFK,PLATFORM_DESC,ORDER_ID,USER_TYPE_SFK,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,20240324,22:47:30,Social,PT,59379341,158243012,1043386,Web,900004671511,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
1,20240415,09:39:00,Paid Search,PT,5546434,10630674,271649,Web,900004734460,1,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
2,20240415,09:39:00,Paid Search,PT,5546434,10630674,271649,Web,900004734493,1,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
3,20240317,17:01:03,Social Ads,PT,33558372,159319069,1075434,Web,900004650781,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
4,20240130,16:12:00,Organic Search,PT,14806064,158837083,324517,Web,900004492878,-1,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79089,20251219,21:39:38,Journeys MC,PT,90023625,162015606,245493687,Web,900008408153,1,2026-10-17 20:45:56,mysql,parfois.sessions_4,scheduled__2026-10-13T05_00_00_00_00,20261017T204556,2026-10-13
79090,20251217,17:51:53,Unassigned,PT,89760227,-1,244986180,Web,900008390903,2,2026-10-17 20:45:56,mysql,parfois.sessions_4,scheduled__2026-10-13T05_00_00_00_00,20261017T204556,2026-10-13
79091,20250909,01:55:18,Organic Search,PT,49799547,906265,211922184,Web,900007709190,1,2026-10-17 20:45:56,mysql,parfois.sessions_4,scheduled__2026-10-13T05_00_00_00_00,20261017T204556,2026-10-13
79092,20250909,01:55:18,Organic Search,PT,49799547,906265,211922184,Web,900007709188,1,2026-10-17 20:45:56,mysql,parfois.sessions_4,scheduled__2026-10-13T05_00_00_00_00,20261017T204556,2026-10-13


In [49]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{order_det}')
WHERE order_id = '900007806632'

""").df()

df_T

,TYPE,ORDER_ID,PROD_SFK,PROD_QTY,MOV_DATE_SFK,AMOUNT_FIXED_RATE_W_DISC_WO_VAT,AMOUNT_FIXED_RATE_WO_DISC_WO_VAT,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date


In [44]:
query_volume_sessoes_orfas = f"""
SELECT 
    DATE_SFK,
    COUNT(*) AS quantidade_sessoes_sem_pedido
FROM read_parquet('{sessions}')
WHERE ORDER_ID IS NOT NULL
  AND CAST(CAST(ORDER_ID AS BIGINT) AS VARCHAR) NOT IN (
      SELECT ORDER_ID 
      FROM read_parquet('{order_cab}')
      WHERE ORDER_ID IS NOT NULL
  )
  AND CAST(CAST(ORDER_ID AS BIGINT) AS VARCHAR) NOT IN (
      SELECT ORDER_ID 
      FROM read_parquet('{order_det}')
      WHERE ORDER_ID IS NOT NULL
  )
GROUP BY DATE_SFK
ORDER BY quantidade_sessoes_sem_pedido DESC, DATE_SFK;
"""

df_volume_sessoes_orfas = con.execute(query_volume_sessoes_orfas).df()
display(df_volume_sessoes_orfas)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,DATE_SFK,quantidade_sessoes_sem_pedido
0,20251126,2371
1,20241129,1160
2,20251128,1022
3,20241128,984
4,20251127,832
...,...,...
726,20250416,34
727,20251015,34
728,20251101,31
729,20241224,28


## Customer ID que não se cruza entre emails e customers

In [57]:
query_emails_issues = f"""
SELECT e.CUSTOMER_ID_FK, e.SENT, e.OPENS, e.CLICKS
FROM read_parquet('{emails}') e
WHERE 
    e.CUSTOMER_ID_FK NOT IN (
        SELECT CUSTOMER_ID 
        FROM read_parquet('{customers}') 
        WHERE CUSTOMER_ID IS NOT NULL
    )
"""
df_emails_issues = con.execute(query_emails_issues).df()
display(df_emails_issues)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,CUSTOMER_ID_FK,SENT,OPENS,CLICKS
0,8162755,2,2.0,NaN
1,161259541,1,1.0,NaN
2,161236777,1,NaN,NaN
3,161241788,1,NaN,NaN
4,161232132,1,NaN,NaN
...,...,...,...,...
900,161257099,1,NaN,NaN
901,161261793,1,NaN,NaN
902,161230060,1,NaN,NaN
903,161237984,1,1.0,NaN


In [61]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{customers}')
WHERE CUSTOMER_ID = '161261403'

""").df()

df_T

,CUSTOMER_ID,CREATION_DATE,SUBSCRIPTION_DATE,UNSUBSCRIPTION_DATE,COUNTRY,BIRTHDAY,SUBSCRIPTION_SOURCE,EMAIL_DOMAIN,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date


## Eventos em que a data de término é menor que a data de início

In [62]:
query_events_issues = f"""
SELECT MARK_COD, EVENT, START_DATE, END_DATE
FROM read_parquet('{events}')
WHERE START_DATE IS NOT NULL 
  AND END_DATE IS NOT NULL 
  AND CAST(END_DATE AS DATE) < CAST(START_DATE AS DATE)
"""
df_events_issues = con.execute(query_events_issues).df()
display(df_events_issues)

,MARK_COD,EVENT,START_DATE,END_DATE
0,Online FR,Avant-Premiere,2025-12-26,2025-01-06
1,Online FI,Clearance,2025-12-26,2025-02-28
2,Online SE,Clearance,2025-12-26,2025-02-28
3,Online DK,Clearance,2025-12-26,2025-02-28
4,Online CH,Clearance,2025-12-26,2025-02-28


In [68]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{events}')
WHERE EVENT = 'Avant-Premiere'

""").df()

df_T

,MARK_COD,MARK_DES,EVENT,START_DATE,END_DATE,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,Online FR,parfois.com,Avant-Premiere,2024-06-11,2024-06-25,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
1,Online FR,parfois.com,Avant-Premiere,2024-12-26,2025-01-07,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
2,Online FR,parfois.com,Avant-Premiere,2025-06-10,2025-06-23,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
3,Online LU,parfois.com,Avant-Premiere,2025-06-10,2025-06-25,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13
4,Online FR,parfois.com,Avant-Premiere,2025-12-26,2025-01-06,2026-10-17 20:49:36,mysql,parfois.events,scheduled__2026-10-13T05_00_00_00_00,20261017T204936,2026-10-13


In [69]:
query_orderdet_issues = f"""
SELECT ORDER_ID, TYPE, PROD_SFK, PROD_QTY, AMOUNT_FIXED_RATE_W_DISC_WO_VAT
FROM read_parquet('{order_det}')
WHERE 
    (TYPE = 'SALE' AND PROD_QTY < 0)
    OR (TYPE = 'RETURN' AND PROD_QTY > 0)
"""
df_orderdet_issues = con.execute(query_orderdet_issues).df()
display(df_orderdet_issues)

,ORDER_ID,TYPE,PROD_SFK,PROD_QTY,AMOUNT_FIXED_RATE_W_DISC_WO_VAT


## Produtos duplicados

In [72]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{product}')

""").df()

df_T

,PROD_SPK,PROD_COD,CAT_DES_EN,GFA_DES_EN,GFS_DES_EN,COLLECTION_TYPE,SEA_COD,ACTIVE_RECORD,FASHIONTYPE,CAT_ACTIVE,TAR_DES,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,6181247,246468_DMM,Apparel,Top,Woven,Core Collection,261,1,Special,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
1,6181248,246468_DMS,Apparel,Top,Woven,Core Collection,261,1,Special,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
2,6181249,246468_RDL,Apparel,Top,Woven,Core Collection,261,1,Special,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
3,6181250,246468_RDM,Apparel,Top,Woven,Core Collection,261,1,Special,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
4,6181251,246468_RDS,Apparel,Top,Woven,Core Collection,261,1,Special,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257665,5798654,205745_BKU,Sunglasses,Round,Plastic,Core Collection,241,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
257666,5798655,205745_KKU,Sunglasses,Round,Plastic,Core Collection,241,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
257667,5798656,205745_TLU,Sunglasses,Round,Plastic,Core Collection,241,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
257668,5798657,2115851BKU,Sunglasses,Butterfly,Plastic,Core Collection,251,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13


In [75]:
query_product_issues = f"""
SELECT PROD_SPK, COUNT(*) as qtd_registros
FROM read_parquet('{product}')
GROUP BY PROD_SPK
HAVING COUNT(*) > 1
ORDER BY qtd_registros DESC
"""
df_product_issues = con.execute(query_product_issues).df()
display(df_product_issues)

,PROD_SPK,qtd_registros


In [76]:
query_product_issues = f"""
SELECT PROD_COD, COUNT(*) as qtd_registros
FROM read_parquet('{product}')
GROUP BY PROD_COD
HAVING COUNT(*) > 1
ORDER BY qtd_registros DESC
"""
df_product_issues = con.execute(query_product_issues).df()
display(df_product_issues)

,PROD_COD,qtd_registros
0,198247_BKXL,53
1,197224_YLM,51
2,197224_TUM,51
3,197224_WTM,51
4,203777_BMXL,50
...,...,...
62963,245993_PKU,2
62964,246003_RDU,2
62965,245999_CAS,2
62966,246002_ECS,2


In [77]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{product}')
where PROD_COD = '198247_BKXL'

""").df()

df_T

,PROD_SPK,PROD_COD,CAT_DES_EN,GFA_DES_EN,GFS_DES_EN,COLLECTION_TYPE,SEA_COD,ACTIVE_RECORD,FASHIONTYPE,CAT_ACTIVE,TAR_DES,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,5813990,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Fashion Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
1,5907797,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
2,5908037,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
3,5929133,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
4,5950284,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
5,5950466,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
6,5950595,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
7,5950989,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
8,5951128,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
9,5951315,198247_BKXL,Hand Bag,Computer Hand Bag,Plain PU,Other Specific,232,0,Basic,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13


## Usuários logados em session que não existem

In [78]:
query_sessions_issues = f"""
SELECT s.SESSION_SFK, s.DATE_SFK, s.CUSTOMER_ID_FK
FROM read_parquet('{sessions}') s
WHERE s.CUSTOMER_ID_FK != -1 
  AND s.CUSTOMER_ID_FK NOT IN (
      SELECT CUSTOMER_ID 
      FROM read_parquet('{customers}') 
      WHERE CUSTOMER_ID IS NOT NULL
  )
"""
df_sessions_issues = con.execute(query_sessions_issues).df()
display(df_sessions_issues)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,SESSION_SFK,DATE_SFK,CUSTOMER_ID_FK
0,31473938,20240526,8162755
1,45667254,20240327,8162755
2,45700532,20240421,8162755
3,46330026,20240518,8162755
4,92582055,20240402,8162755
...,...,...,...
285,205407042,20250817,161239086
286,246186922,20251220,161261050
287,249967415,20251230,161261050
288,213715846,20250915,161256622


In [83]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{customers}')
where Customer_ID = '8162755'

""").df()

df_T

,CUSTOMER_ID,CREATION_DATE,SUBSCRIPTION_DATE,UNSUBSCRIPTION_DATE,COUNTRY,BIRTHDAY,SUBSCRIPTION_SOURCE,EMAIL_DOMAIN,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date


In [85]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{sessions}')
where Customer_ID_FK = '8162755'

""").df()

df_T

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,DATE_SFK,TIMESTAMP,CHANNEL_DESC,COUNTRY_COD,CLIENT_SFK,CUSTOMER_ID_FK,SESSION_SFK,PLATFORM_DESC,ORDER_ID,USER_TYPE_SFK,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,20240420,08:47:14,Direct,PL,48301374,8162755,38070023,APP,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
1,20240521,03:05:18,Direct,PL,48301374,8162755,35389160,APP,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
2,20240419,03:36:12,Direct,PL,48301374,8162755,50132803,APP,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
3,20240526,03:00:00,Paid Search,PL,48301374,8162755,62885223,APP,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
4,20240317,11:14:05,Direct,PL,48301374,8162755,112415873,APP,NaN,2,2026-10-17 20:39:29,mysql,parfois.sessions_1,scheduled__2026-10-13T05_00_00_00_00,20261017T203929,2026-10-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,20250104,01:44:03,Direct,PL,48301374,8162755,45788417,APP,NaN,1,2026-10-17 20:43:13,mysql,parfois.sessions_3,scheduled__2026-10-13T05_00_00_00_00,20261017T204313,2026-10-13
85,20250126,14:58:10,Direct,PL,1878709,8162755,132710841,APP,NaN,1,2026-10-17 20:43:13,mysql,parfois.sessions_3,scheduled__2026-10-13T05_00_00_00_00,20261017T204313,2026-10-13
86,20250201,19:47:22,Direct,PL,1878709,8162755,19646933,APP,NaN,1,2026-10-17 20:43:13,mysql,parfois.sessions_3,scheduled__2026-10-13T05_00_00_00_00,20261017T204313,2026-10-13
87,20250311,17:39:57,Direct,PL,1878709,8162755,147957661,APP,NaN,1,2026-10-17 20:43:13,mysql,parfois.sessions_3,scheduled__2026-10-13T05_00_00_00_00,20261017T204313,2026-10-13


# Nulos


In [89]:

# ==========================================
# 0. TABELA: Customers
# ==========================================
query_detalhe_problemas = f"""
SELECT 
    column_name,
    (count(CASE WHEN value IS NULL THEN 1 END) * 100.0 / count(*)) AS pct_nulos,
    (count(CASE WHEN value = 'NaN' OR value = 'nan' OR value = 'NAN' THEN 1 END) * 100.0 / count(*)) AS pct_nan,
    (count(CASE WHEN value = '-1' OR value = '-1.0' THEN 1 END) * 100.0 / count(*)) AS pct_menos_um
FROM (
    SELECT 'CUSTOMER_ID' AS column_name, CAST(CUSTOMER_ID AS VARCHAR) AS value FROM read_parquet('{customers}')
    UNION ALL SELECT 'CREATION_DATE', CAST(CREATION_DATE AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT 'SUBSCRIPTION_DATE', CAST(SUBSCRIPTION_DATE AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT 'UNSUBSCRIPTION_DATE', CAST(UNSUBSCRIPTION_DATE AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT 'COUNTRY', CAST(COUNTRY AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT 'BIRTHDAY', CAST(BIRTHDAY AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT 'SUBSCRIPTION_SOURCE', CAST(SUBSCRIPTION_SOURCE AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT 'EMAIL_DOMAIN', CAST(EMAIL_DOMAIN AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT '_ingested_at', CAST(_ingested_at AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT '_source_system', CAST(_source_system AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT '_source_entity', CAST(_source_entity AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT '_batch_id', CAST(_batch_id AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT '_load_id', CAST(_load_id AS VARCHAR) FROM read_parquet('{customers}')
    UNION ALL SELECT 'ingestion_date', CAST(ingestion_date AS VARCHAR) FROM read_parquet('{customers}')
) AS unpivot_data
GROUP BY column_name
ORDER BY pct_nulos DESC;
"""

df_detalhe_problemas = con.execute(query_detalhe_problemas).df()
display(df_detalhe_problemas)



# ==========================================
# 1. TABELA: EMAILS
# ==========================================
query_emails_problemas = f"""
SELECT 
    column_name,
    (count(CASE WHEN value IS NULL THEN 1 END) * 100.0 / count(*)) AS pct_nulos,
    (count(CASE WHEN value = 'NaN' OR value = 'nan' OR value = 'NAN' THEN 1 END) * 100.0 / count(*)) AS pct_nan,
    (count(CASE WHEN value = '-1' OR value = '-1.0' THEN 1 END) * 100.0 / count(*)) AS pct_menos_um
FROM (
    SELECT 'CUSTOMER_ID_FK' AS column_name, CAST(CUSTOMER_ID_FK AS VARCHAR) AS value FROM read_parquet('{emails}')
    UNION ALL SELECT 'SEND_DATE_SFK', CAST(SEND_DATE_SFK AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT 'SENT', CAST(SENT AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT 'OPENS', CAST(OPENS AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT 'CLICKS', CAST(CLICKS AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT 'UNSUBS', CAST(UNSUBS AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT '_ingested_at', CAST(_ingested_at AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT '_source_system', CAST(_source_system AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT '_source_entity', CAST(_source_entity AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT '_batch_id', CAST(_batch_id AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT '_load_id', CAST(_load_id AS VARCHAR) FROM read_parquet('{emails}')
    UNION ALL SELECT 'ingestion_date', CAST(ingestion_date AS VARCHAR) FROM read_parquet('{emails}')
) AS unpivot_data
GROUP BY column_name
ORDER BY pct_nulos DESC;
"""
df_emails_problemas = con.execute(query_emails_problemas).df()
display(df_emails_problemas)


# ==========================================
# 2. TABELA: EVENTS
# ==========================================
query_events_problemas = f"""
SELECT 
    column_name,
    (count(CASE WHEN value IS NULL THEN 1 END) * 100.0 / count(*)) AS pct_nulos,
    (count(CASE WHEN value = 'NaN' OR value = 'nan' OR value = 'NAN' THEN 1 END) * 100.0 / count(*)) AS pct_nan,
    (count(CASE WHEN value = '-1' OR value = '-1.0' THEN 1 END) * 100.0 / count(*)) AS pct_menos_um
FROM (
    SELECT 'MARK_COD' AS column_name, CAST(MARK_COD AS VARCHAR) AS value FROM read_parquet('{events}')
    UNION ALL SELECT 'MARK_DES', CAST(MARK_DES AS VARCHAR) FROM read_parquet('{events}')
    UNION ALL SELECT 'EVENT', CAST(EVENT AS VARCHAR) FROM read_parquet('{events}')
    UNION ALL SELECT 'START_DATE', CAST(START_DATE AS VARCHAR) FROM read_parquet('{events}')
    UNION ALL SELECT 'END_DATE', CAST(END_DATE AS VARCHAR) FROM read_parquet('{events}')
    UNION ALL SELECT '_ingested_at', CAST(_ingested_at AS VARCHAR) FROM read_parquet('{events}')
    UNION ALL SELECT '_source_system', CAST(_source_system AS VARCHAR) FROM read_parquet('{events}')
    UNION ALL SELECT '_source_entity', CAST(_source_entity AS VARCHAR) FROM read_parquet('{events}')
    UNION ALL SELECT '_batch_id', CAST(_batch_id AS VARCHAR) FROM read_parquet('{events}')
    UNION ALL SELECT '_load_id', CAST(_load_id AS VARCHAR) FROM read_parquet('{events}')
    UNION ALL SELECT 'ingestion_date', CAST(ingestion_date AS VARCHAR) FROM read_parquet('{events}')
) AS unpivot_data
GROUP BY column_name
ORDER BY pct_nulos DESC;
"""
df_events_problemas = con.execute(query_events_problemas).df()
display(df_events_problemas)


# ==========================================
# 3. TABELA: ORDER_CAB
# ==========================================
query_ordercab_problemas = f"""
SELECT 
    column_name,
    (count(CASE WHEN value IS NULL THEN 1 END) * 100.0 / count(*)) AS pct_nulos,
    (count(CASE WHEN value = 'NaN' OR value = 'nan' OR value = 'NAN' THEN 1 END) * 100.0 / count(*)) AS pct_nan,
    (count(CASE WHEN value = '-1' OR value = '-1.0' THEN 1 END) * 100.0 / count(*)) AS pct_menos_um
FROM (
    SELECT 'ORDER_ID' AS column_name, CAST(ORDER_ID AS VARCHAR) AS value FROM read_parquet('{order_cab}')
    UNION ALL SELECT 'ORDER_DATE', CAST(ORDER_DATE AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT 'ORDER_ORIGIN', CAST(ORDER_ORIGIN AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT 'ORDER_CHANEL_TYPE', CAST(ORDER_CHANEL_TYPE AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT 'CUSTOMER_TYPE', CAST(CUSTOMER_TYPE AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT 'CUSTOMER_ID_FK', CAST(CUSTOMER_ID_FK AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT 'MARK_COD', CAST(MARK_COD AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT 'FULFILLMENT_STATUS_ID', CAST(FULFILLMENT_STATUS_ID AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT '_ingested_at', CAST(_ingested_at AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT '_source_system', CAST(_source_system AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT '_source_entity', CAST(_source_entity AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT '_batch_id', CAST(_batch_id AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT '_load_id', CAST(_load_id AS VARCHAR) FROM read_parquet('{order_cab}')
    UNION ALL SELECT 'ingestion_date', CAST(ingestion_date AS VARCHAR) FROM read_parquet('{order_cab}')
) AS unpivot_data
GROUP BY column_name
ORDER BY pct_nulos DESC;
"""
df_ordercab_problemas = con.execute(query_ordercab_problemas).df()
display(df_ordercab_problemas)


# ==========================================
# 4. TABELA: ORDER_DET
# ==========================================
query_orderdet_problemas = f"""
SELECT 
    column_name,
    (count(CASE WHEN value IS NULL THEN 1 END) * 100.0 / count(*)) AS pct_nulos,
    (count(CASE WHEN value = 'NaN' OR value = 'nan' OR value = 'NAN' THEN 1 END) * 100.0 / count(*)) AS pct_nan,
    (count(CASE WHEN value = '-1' OR value = '-1.0' THEN 1 END) * 100.0 / count(*)) AS pct_menos_um
FROM (
    SELECT 'TYPE' AS column_name, CAST(TYPE AS VARCHAR) AS value FROM read_parquet('{order_det}')
    UNION ALL SELECT 'ORDER_ID', CAST(ORDER_ID AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT 'PROD_SFK', CAST(PROD_SFK AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT 'PROD_QTY', CAST(PROD_QTY AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT 'MOV_DATE_SFK', CAST(MOV_DATE_SFK AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT 'AMOUNT_FIXED_RATE_W_DISC_WO_VAT', CAST(AMOUNT_FIXED_RATE_W_DISC_WO_VAT AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT 'AMOUNT_FIXED_RATE_WO_DISC_WO_VAT', CAST(AMOUNT_FIXED_RATE_WO_DISC_WO_VAT AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT '_ingested_at', CAST(_ingested_at AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT '_source_system', CAST(_source_system AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT '_source_entity', CAST(_source_entity AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT '_batch_id', CAST(_batch_id AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT '_load_id', CAST(_load_id AS VARCHAR) FROM read_parquet('{order_det}')
    UNION ALL SELECT 'ingestion_date', CAST(ingestion_date AS VARCHAR) FROM read_parquet('{order_det}')
) AS unpivot_data
GROUP BY column_name
ORDER BY pct_nulos DESC;
"""
df_orderdet_problemas = con.execute(query_orderdet_problemas).df()
display(df_orderdet_problemas)


# ==========================================
# 5. TABELA: PAID_MEDIA
# ==========================================
query_paidmedia_problemas = f"""
SELECT 
    column_name,
    (count(CASE WHEN value IS NULL THEN 1 END) * 100.0 / count(*)) AS pct_nulos,
    (count(CASE WHEN value = 'NaN' OR value = 'nan' OR value = 'NAN' THEN 1 END) * 100.0 / count(*)) AS pct_nan,
    (count(CASE WHEN value = '-1' OR value = '-1.0' THEN 1 END) * 100.0 / count(*)) AS pct_menos_um
FROM (
    SELECT 'DATE_SFK' AS column_name, CAST(DATE_SFK AS VARCHAR) AS value FROM read_parquet('{paid_media}')
    UNION ALL SELECT 'MARKETPLACE', CAST(MARKETPLACE AS VARCHAR) FROM read_parquet('{paid_media}')
    UNION ALL SELECT 'AMOUNT_PAID_MEDIA', CAST(AMOUNT_PAID_MEDIA AS VARCHAR) FROM read_parquet('{paid_media}')
    UNION ALL SELECT '_ingested_at', CAST(_ingested_at AS VARCHAR) FROM read_parquet('{paid_media}')
    UNION ALL SELECT '_source_system', CAST(_source_system AS VARCHAR) FROM read_parquet('{paid_media}')
    UNION ALL SELECT '_source_entity', CAST(_source_entity AS VARCHAR) FROM read_parquet('{paid_media}')
    UNION ALL SELECT '_batch_id', CAST(_batch_id AS VARCHAR) FROM read_parquet('{paid_media}')
    UNION ALL SELECT '_load_id', CAST(_load_id AS VARCHAR) FROM read_parquet('{paid_media}')
    UNION ALL SELECT 'ingestion_date', CAST(ingestion_date AS VARCHAR) FROM read_parquet('{paid_media}')
) AS unpivot_data
GROUP BY column_name
ORDER BY pct_nulos DESC;
"""
df_paidmedia_problemas = con.execute(query_paidmedia_problemas).df()
display(df_paidmedia_problemas)


# ==========================================
# 6. TABELA: PRODUCT
# ==========================================
query_product_problemas = f"""
SELECT 
    column_name,
    (count(CASE WHEN value IS NULL THEN 1 END) * 100.0 / count(*)) AS pct_nulos,
    (count(CASE WHEN value = 'NaN' OR value = 'nan' OR value = 'NAN' THEN 1 END) * 100.0 / count(*)) AS pct_nan,
    (count(CASE WHEN value = '-1' OR value = '-1.0' THEN 1 END) * 100.0 / count(*)) AS pct_menos_um
FROM (
    SELECT 'PROD_SPK' AS column_name, CAST(PROD_SPK AS VARCHAR) AS value FROM read_parquet('{product}')
    UNION ALL SELECT 'PROD_COD', CAST(PROD_COD AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'CAT_DES_EN', CAST(CAT_DES_EN AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'GFA_DES_EN', CAST(GFA_DES_EN AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'GFS_DES_EN', CAST(GFS_DES_EN AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'COLLECTION_TYPE', CAST(COLLECTION_TYPE AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'SEA_COD', CAST(SEA_COD AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'ACTIVE_RECORD', CAST(ACTIVE_RECORD AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'FASHIONTYPE', CAST(FASHIONTYPE AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'CAT_ACTIVE', CAST(CAT_ACTIVE AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'TAR_DES', CAST(TAR_DES AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT '_ingested_at', CAST(_ingested_at AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT '_source_system', CAST(_source_system AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT '_source_entity', CAST(_source_entity AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT '_batch_id', CAST(_batch_id AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT '_load_id', CAST(_load_id AS VARCHAR) FROM read_parquet('{product}')
    UNION ALL SELECT 'ingestion_date', CAST(ingestion_date AS VARCHAR) FROM read_parquet('{product}')
) AS unpivot_data
GROUP BY column_name
ORDER BY pct_nulos DESC;
"""
df_product_problemas = con.execute(query_product_problemas).df()
display(df_product_problemas)


# ==========================================
# 7. TABELA: SESSIONS
# ==========================================
query_sessions_problemas = f"""
SELECT 
    column_name,
    (count(CASE WHEN value IS NULL THEN 1 END) * 100.0 / count(*)) AS pct_nulos,
    (count(CASE WHEN value = 'NaN' OR value = 'nan' OR value = 'NAN' THEN 1 END) * 100.0 / count(*)) AS pct_nan,
    (count(CASE WHEN value = '-1' OR value = '-1.0' THEN 1 END) * 100.0 / count(*)) AS pct_menos_um
FROM (
    SELECT 'DATE_SFK' AS column_name, CAST(DATE_SFK AS VARCHAR) AS value FROM read_parquet('{sessions}')
    UNION ALL SELECT 'TIMESTAMP', CAST(TIMESTAMP AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT 'CHANNEL_DESC', CAST(CHANNEL_DESC AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT 'COUNTRY_COD', CAST(COUNTRY_COD AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT 'CLIENT_SFK', CAST(CLIENT_SFK AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT 'CUSTOMER_ID_FK', CAST(CUSTOMER_ID_FK AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT 'SESSION_SFK', CAST(SESSION_SFK AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT 'PLATFORM_DESC', CAST(PLATFORM_DESC AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT 'ORDER_ID', CAST(ORDER_ID AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT 'USER_TYPE_SFK', CAST(USER_TYPE_SFK AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT '_ingested_at', CAST(_ingested_at AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT '_source_system', CAST(_source_system AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT '_source_entity', CAST(_source_entity AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT '_batch_id', CAST(_batch_id AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT '_load_id', CAST(_load_id AS VARCHAR) FROM read_parquet('{sessions}')
    UNION ALL SELECT 'ingestion_date', CAST(ingestion_date AS VARCHAR) FROM read_parquet('{sessions}')
) AS unpivot_data
GROUP BY column_name
ORDER BY pct_nulos DESC;
"""
df_sessions_problemas = con.execute(query_sessions_problemas).df()
display(df_sessions_problemas)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,pct_nulos,pct_nan,pct_menos_um
0,UNSUBSCRIPTION_DATE,90.392440,0.0,0.0
1,BIRTHDAY,79.214197,0.0,0.0
2,SUBSCRIPTION_SOURCE,56.569658,0.0,0.0
3,SUBSCRIPTION_DATE,49.835116,0.0,0.0
4,COUNTRY,8.398455,0.0,0.0
5,CREATION_DATE,0.868763,0.0,0.0
6,EMAIL_DOMAIN,0.564192,0.0,0.0
7,ingestion_date,0.000000,0.0,0.0
8,_batch_id,0.000000,0.0,0.0
9,_source_entity,0.000000,0.0,0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,pct_nulos,pct_nan,pct_menos_um
0,UNSUBS,99.873730,0.0,0.0
1,CLICKS,98.358254,0.0,0.0
2,OPENS,73.955474,0.0,0.0
3,CUSTOMER_ID_FK,0.001705,0.0,0.0
4,_source_system,0.000000,0.0,0.0
5,ingestion_date,0.000000,0.0,0.0
6,_load_id,0.000000,0.0,0.0
7,_batch_id,0.000000,0.0,0.0
8,SEND_DATE_SFK,0.000000,0.0,0.0
9,_ingested_at,0.000000,0.0,0.0


,column_name,pct_nulos,pct_nan,pct_menos_um
0,MARK_COD,3.950617,0.0,0.0
1,_batch_id,0.000000,0.0,0.0
2,ingestion_date,0.000000,0.0,0.0
3,_load_id,0.000000,0.0,0.0
4,EVENT,0.000000,0.0,0.0
5,END_DATE,0.000000,0.0,0.0
6,_ingested_at,0.000000,0.0,0.0
7,MARK_DES,0.000000,0.0,0.0
8,_source_entity,0.000000,0.0,0.0
9,_source_system,0.000000,0.0,0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (3 outputs left out)

,column_name,pct_nulos,pct_nan,pct_menos_um
0,AMOUNT_PAID_MEDIA,0.0,0.0,0.0
1,DATE_SFK,0.0,0.0,0.0
2,ingestion_date,0.0,0.0,0.0
3,_batch_id,0.0,0.0,0.0
4,_source_entity,0.0,0.0,0.0
5,_source_system,0.0,0.0,0.0
6,_load_id,0.0,0.0,0.0
7,_ingested_at,0.0,0.0,0.0
8,MARKETPLACE,0.0,0.0,0.0


,column_name,pct_nulos,pct_nan,pct_menos_um
0,COLLECTION_TYPE,3.658167,0.0,0.0
1,TAR_DES,0.154849,0.0,0.0
2,GFS_DES_EN,0.107890,0.0,0.0
3,GFA_DES_EN,0.107890,0.0,0.0
4,CAT_DES_EN,0.107890,0.0,0.0
5,_source_entity,0.000000,0.0,0.0
6,PROD_COD,0.000000,0.0,0.0
7,ingestion_date,0.000000,0.0,0.0
8,PROD_SPK,0.000000,0.0,0.0
9,_load_id,0.000000,0.0,0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,pct_nulos,pct_nan,pct_menos_um
0,ORDER_ID,98.672264,0.0,0.000000
1,_source_system,0.000000,0.0,0.000000
2,USER_TYPE_SFK,0.000000,0.0,3.269874
3,SESSION_SFK,0.000000,0.0,0.000000
4,DATE_SFK,0.000000,0.0,0.000000
5,_source_entity,0.000000,0.0,0.000000
6,CLIENT_SFK,0.000000,0.0,0.000000
7,_ingested_at,0.000000,0.0,0.000000
8,PLATFORM_DESC,0.000000,0.0,0.000000
9,CUSTOMER_ID_FK,0.000000,0.0,78.628026


In [91]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{product}')
where TAR_DES is NULL

""").df()

df_T

,PROD_SPK,PROD_COD,CAT_DES_EN,GFA_DES_EN,GFS_DES_EN,COLLECTION_TYPE,SEA_COD,ACTIVE_RECORD,FASHIONTYPE,CAT_ACTIVE,TAR_DES,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,6182772,241959,NaN,NaN,NaN,Core Collection,261,0,Fashion,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
1,6185470,247577,INT - BARAM,Hardware Acquision,Computadores,NaN,252,1,NÃO DEFINIDO,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
2,6185471,247578,INT - BARAM,Hardware Acquision,Monitores,NaN,252,1,NÃO DEFINIDO,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
3,6185836,234494,NaN,NaN,NaN,Core Collection,252,1,Fashion Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
4,6185838,234501,NaN,NaN,NaN,Core Collection,252,1,Fashion Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,5784908,211826,INT - BARAM,Hardware Acquision,Video-vigilância,NaN,232,1,NÃO DEFINIDO,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
395,5784909,211829,INT - BARAM,Hardware Acquision,Video-vigilância,NaN,232,1,NÃO DEFINIDO,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
396,5796104,212402_BMU,NaN,NaN,NaN,NaN,232,0,NÃO DEFINIDO,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
397,5796127,212402_BMU,NaN,NaN,NaN,NaN,232,1,NÃO DEFINIDO,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13


In [104]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{product}')
where FASHIONTYPE = 'NÃO DEFINIDO'

""").df()

df_T

,PROD_SPK,PROD_COD,CAT_DES_EN,GFA_DES_EN,GFS_DES_EN,COLLECTION_TYPE,SEA_COD,ACTIVE_RECORD,FASHIONTYPE,CAT_ACTIVE,TAR_DES,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,6181362,246396_BKS,Night,Shoulder Bag,Fantasy PU,Core Collection,261,0,NÃO DEFINIDO,Y,Finished Product,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
1,6181452,122294_WTND,Works,Display Acessories,Plastic Hangers,NaN,251,1,NÃO DEFINIDO,N,Works,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
2,6181464,194579_NCND,Works,Exposition,Hangers,NaN,251,1,NÃO DEFINIDO,N,Works,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
3,6181465,238304,Consumable,Internal Use,Warehouse utilities,NaN,251,1,NÃO DEFINIDO,N,Consumable,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
4,6181473,242112_001,Consumable,Sale,Paper bags,NaN,251,1,NÃO DEFINIDO,N,Packs,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8251,5797217,214348,Informática,Hardware Acquision,POS (e componentes),NaN,232,1,NÃO DEFINIDO,N,Informática,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
8252,5797218,214349,Informática,Hardware Acquision,Componentes,NaN,232,1,NÃO DEFINIDO,N,Informática,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
8253,5797272,214373,INT - BARAM,Hardware Acquision,Video-vigilância,NaN,232,1,NÃO DEFINIDO,N,NaN,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
8254,5797343,214376,Informática,Hardware Acquision,Computadores,NaN,232,1,NÃO DEFINIDO,N,Informática,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13


In [99]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{product}')
where GFS_DES_EN is NULL

""").df()

df_T

,PROD_SPK,PROD_COD,CAT_DES_EN,GFA_DES_EN,GFS_DES_EN,COLLECTION_TYPE,SEA_COD,ACTIVE_RECORD,FASHIONTYPE,CAT_ACTIVE,TAR_DES,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,6182772,241959,None,None,None,Core Collection,261,0,Fashion,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
1,6185836,234494,None,None,None,Core Collection,252,1,Fashion Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
2,6185838,234501,None,None,None,Core Collection,252,1,Fashion Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
3,6185840,235173,None,None,None,Core Collection,252,1,Fashion Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
4,6185842,237636,None,None,None,Core Collection,252,1,Fashion Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,6165234,239493,None,None,None,Set,252,1,Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
274,6165253,239502,None,None,None,Set,252,1,Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
275,6165761,244505,None,None,None,Exclusive Online,261,1,Fashion Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
276,6178207,241959,None,None,None,Core Collection,261,0,Fashion,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13


In [103]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{product}')
where CAT_DES_EN is NULL

""").df()

df_T.iloc[50:70]

,PROD_SPK,PROD_COD,CAT_DES_EN,GFA_DES_EN,GFS_DES_EN,COLLECTION_TYPE,SEA_COD,ACTIVE_RECORD,FASHIONTYPE,CAT_ACTIVE,TAR_DES,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
50,5965203,237651,None,None,None,Core Collection,252,0,Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
51,5975792,240144,None,None,None,Core Collection,252,0,Plus Closet,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
52,5976042,240655,None,None,None,Core Collection,261,0,Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
53,5976115,240816,None,None,None,Core Collection,252,0,Fashion Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
54,5976327,241142,None,None,None,Core Collection,261,0,Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
55,5976331,241147,None,None,None,Core Collection,261,0,Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
56,5976337,241153,None,None,None,Core Collection,261,0,Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
57,5976355,241183,None,None,None,Online + Lojas Imagem,252,0,Capsule,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
58,5976367,241206,None,None,None,Core Collection,261,0,Basic,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13
59,5976911,241922,None,None,None,Core Collection,261,0,Fashion,N,None,2026-10-17 20:52:02,mysql,parfois.product,scheduled__2026-10-13T05_00_00_00_00,20261017T205202,2026-10-13


In [92]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{order_det}')
where PROD_SFK = '6185470'

""").df()

df_T

,TYPE,ORDER_ID,PROD_SFK,PROD_QTY,MOV_DATE_SFK,AMOUNT_FIXED_RATE_W_DISC_WO_VAT,AMOUNT_FIXED_RATE_WO_DISC_WO_VAT,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date


In [106]:
# Columns and types of a dataset
df_T = con.sql(f"""

Select *
FROM read_parquet('{order_det}')
Limit(111)

""").df()

df_T

,TYPE,ORDER_ID,PROD_SFK,PROD_QTY,MOV_DATE_SFK,AMOUNT_FIXED_RATE_W_DISC_WO_VAT,AMOUNT_FIXED_RATE_WO_DISC_WO_VAT,_ingested_at,_source_system,_source_entity,_batch_id,_load_id,ingestion_date
0,SALE,900004669063,5841086,1.0,20240324,5.449560,5.449560,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
1,SALE,900004669071,5841333,1.0,20240324,17.846280,19.829760,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
2,SALE,900004669073,5854690,1.0,20240324,13.383480,14.871060,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
3,RETURN,900004668759,5818961,-1.0,20240410,-17.846280,-17.846280,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
4,RETURN,900004669086,5863652,-1.0,20240403,-19.829760,-19.829760,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,RETURN,900004949752,5858553,-1.0,20240625,-13.879320,-13.879320,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
107,RETURN,900004949823,5853254,-1.0,20240621,-2.970240,-2.970240,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
108,SALE,900004949734,5873910,1.0,20240609,3.961980,3.961980,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
109,SALE,900004949739,5856960,1.0,20240609,3.435712,3.435712,2026-10-17 20:50:52,mysql,parfois.order_det,scheduled__2026-10-13T05_00_00_00_00,20261017T205052,2026-10-13
